# 03 · Embedding + retrieval (Cohere Embed v4 → Pinecone)

**Goal:** embed the corpus, store it in Pinecone, run real searches, and look at the **similarity scores**.
The score distributions for answerable vs unanswerable questions are how we choose the "I don't know" threshold later.

Uses the production clients (`app/clients/`), so timeouts, retries and batching are exercised here too.
Everything is stored in a **scratch namespace** (`nb3-scratch`), separate from real data, and cleaned up at the end.

> Cells use top-level `await`, which Jupyter supports. This notebook makes real (small, billable) API calls.

In [ ]:
import sys, pathlib, time, asyncio, statistics
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from app.config import get_settings
from app.errors import AppError
from app.ingestion.validate import validate_upload
from app.ingestion.extract import extract_text
from app.ingestion.chunking import chunk_document, get_token_counter
from app.ingestion.records import build_records
from app.clients.cohere_client import EmbeddingClient
from app.clients.pinecone_client import VectorStore

settings = get_settings()
embedder = EmbeddingClient(settings)
store = VectorStore(settings)
NAMESPACE = "nb3-scratch"
print(f"model={settings.embedding_model} dim={settings.embedding_dim} "
      f"chunks={settings.chunk_size_tokens}/{settings.chunk_overlap_tokens} index={settings.pinecone_index}")

## 1. Chunk the corpus (config from `.env` / settings)

In [ ]:
entries = []  # (document_id, filename, [chunks])
for path in sorted((ROOT / "corpus").iterdir()):
    if path.suffix.lower() not in {".pdf", ".docx"}:
        continue
    data = path.read_bytes()
    try:
        up = validate_upload(path.name, data, settings.max_upload_mb * 1024 * 1024)
        doc = extract_text(up.file_type, data, max_pages=settings.max_pdf_pages)
    except AppError as exc:
        print("skipped", path.name, "->", exc.message); continue
    chunks = chunk_document(doc, size=settings.chunk_size_tokens, overlap=settings.chunk_overlap_tokens)
    entries.append((up.sha256[:12], up.filename, chunks))
    print(f"{up.filename:40} {len(chunks):>4} chunks")
print("total chunks:", sum(len(c) for _, _, c in entries))

## 2. Embed all chunks (parallel batches)

In [ ]:
texts = [c.text for _, _, chunks in entries for c in chunks]
start = time.perf_counter()
vectors = await embedder.embed_documents(texts)
elapsed = time.perf_counter() - start
print(f"embedded {len(vectors)} chunks in {elapsed:.1f}s  (dim={len(vectors[0])})")
assert len(vectors) == len(texts) and len(vectors[0]) == settings.embedding_dim

## 3. Upsert to Pinecone and wait until searchable
Pinecone serverless is *eventually consistent*: a just-written vector can take a moment to appear. The ingestion service must wait for this before returning success.

In [ ]:
records, offset = [], 0
for document_id, filename, chunks in entries:
    records += build_records(document_id, filename, chunks, vectors[offset:offset + len(chunks)])
    offset += len(chunks)

start = time.perf_counter()
await store.upsert(records, namespace=NAMESPACE)
visible = await store.wait_until_visible(records[-1], namespace=NAMESPACE)
print(f"upserted {len(records)} vectors; searchable={visible}; took {time.perf_counter() - start:.1f}s")

## 4. Search helper

In [ ]:
async def search(question: str, k: int = 5):
    return await store.query(await embedder.embed_query(question), top_k=k, namespace=NAMESPACE)

def where(md: dict) -> str:
    return f"p.{int(md['page'])}" if "page" in md else f"[{md.get('section', '?')}]"

def show(question: str, matches) -> None:
    print(f"\nQ: {question}")
    if not matches:
        print("   (no matches)"); return
    for rank, m in enumerate(matches, 1):
        md_ = m.metadata
        print(f"  {rank}. score={m.score:.3f}  {md_['filename']} {where(md_)}  {md_['text'][:80]!r}")

# EDIT: three questions your documents DO answer
DEMO_QUESTIONS = [
    "EDIT ME: a question your documents answer",
    "EDIT ME: a reworded version of the same question",
    "EDIT ME: a multi-part question",
]
for q in DEMO_QUESTIONS:
    show(q, await search(q, 5))

Check by eye: is the right passage rank 1? Are scores clearly higher for good matches? Note the filename + page of the *correct* source for each question; those labels become your golden set.

## 5. Answerable vs unanswerable: where do the scores separate?

In [ ]:
# EDIT: 6-8 questions per list, written from your corpus
ANSWERABLE = [
    "EDIT ME: question the documents answer #1",
    "EDIT ME: question the documents answer #2",
    "EDIT ME: question the documents answer #3",
]
UNANSWERABLE = [
    "EDIT ME: a question about something NOT in the documents #1",
    "EDIT ME: a question about something NOT in the documents #2",
    "EDIT ME: what is the weather like in Paris today?",
]

async def top_scores(questions):
    results = await asyncio.gather(*(search(q, 5) for q in questions))
    return [(q, m[0].score if m else 0.0) for q, m in zip(questions, results)]

ans, unans = await top_scores(ANSWERABLE), await top_scores(UNANSWERABLE)
print("ANSWERABLE (top-1 score)")
for q, s in ans: print(f"  {s:.3f}  {q[:70]}")
print("UNANSWERABLE (top-1 score)")
for q, s in unans: print(f"  {s:.3f}  {q[:70]}")

a, u = [s for _, s in ans], [s for _, s in unans]
print(f"\nanswerable   min={min(a):.3f} median={statistics.median(a):.3f}")
print(f"unanswerable max={max(u):.3f} median={statistics.median(u):.3f}")
if min(a) > max(u):
    print(f"clean gap -> candidate threshold ~ {(min(a) + max(u)) / 2:.3f}")
else:
    print("scores OVERLAP: a single threshold will misclassify some questions.")
    print("-> this is why the final design also uses reranking and an LLM 'answerable' check.")

**Reading this honestly:** with only ~6 questions per side this is a *rough* picture. The final threshold is calibrated on the full golden set (25+ questions) and must be re-calibrated whenever the embedding model, dimension, chunking or reranking changes. Raw cosine scores are not probabilities.

## 6. Score spread for one query
A steep drop after rank 1 means a confident match; a flat list means the corpus has nothing specific.

In [ ]:
q = ANSWERABLE[0]
matches = await search(q, 10)
print(q)
for rank, m in enumerate(matches, 1):
    print(f"{rank:>2}. {m.score:.3f} {'#' * int(m.score * 40)}")
if len(matches) >= 5:
    print(f"gap rank1 - rank5 = {matches[0].score - matches[4].score:.3f}")

## 7. Clean up the scratch namespace

In [ ]:
CLEANUP = False   # set True when you are finished with this notebook
if CLEANUP:
    await store.delete_namespace(NAMESPACE)
    print("scratch namespace deleted")
else:
    print("left in place (set CLEANUP = True to delete)")

## 8. Notes for DESIGN.md (fill in)
- Embedding time for the corpus and per-query latency: ...
- Did rank 1 contain the right passage? Where did it fail (tables, rewording, multi-part)? ...
- Answerable vs unanswerable score ranges: ...
- Observed eventual-consistency delay before vectors were searchable: ...
- Cost estimate (tokens embedded x price): ...